In [23]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
import time
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)
!ollama pull llama3

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [24]:
from google.colab import files
uploaded = files.upload()

Saving chroma_db_tfm.zip to chroma_db_tfm.zip
Saving columnas_tfm.pkl to columnas_tfm.pkl
Saving modelo_final_tfm.pkl to modelo_final_tfm.pkl


In [25]:
!pip uninstall -y transformers tokenizers sentence-transformers langchain-huggingface
!pip install -qU fastembed langchain-community langchain-ollama chromadb langchain joblib

Found existing installation: transformers 4.40.0
Uninstalling transformers-4.40.0:
  Successfully uninstalled transformers-4.40.0
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
Found existing installation: sentence-transformers 2.7.0
Uninstalling sentence-transformers-2.7.0:
  Successfully uninstalled sentence-transformers-2.7.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 22.2 MB/s eta 0:00:00


In [26]:
import joblib
scaler = joblib.load('columnas_tfm.pkl')
columnas_requeridas = list(scaler)
print(f"Columnas detectadas: {columnas_requeridas}")

Columnas detectadas: ['total_num_interes', 'finca_Total fincas', 'finca_Viviendas', 'tipo_Fijo', 'tipo_Total', 'tipo_Variable', 'interes_bce']


In [27]:
import os, shutil
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# 1. Limpiar base de datos
if os.path.exists("./chroma_db_tfm"):
    !chmod -R 777 ./chroma_db_tfm
    shutil.rmtree("./chroma_db_tfm")

# 2. Descomprimir
!unzip -o chroma_db_tfm.zip -d chroma_db_tfm
!chmod -R 777 ./chroma_db_tfm

# 3. Inicializar Embeddings con FastEmbed
print("Cargando motor de búsqueda ligero...")
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

vector_db = Chroma(
    persist_directory="./chroma_db_tfm",
    embedding_function=embeddings
)

# 4. Inyectar conocimiento del ML
conocimiento_ml = f"RESUMEN TÉCNICO MODELO ML: Variables reales del modelo: {columnas_requeridas}."
doc_ml = Document(page_content=conocimiento_ml, metadata={"source": "config_ml"})

vector_db.add_documents([doc_ml])
print("Sistema cargado con éxito usando FastEmbed.")

Archive:  chroma_db_tfm.zip
   creating: chroma_db_tfm/71307125-5851-4ef3-957c-a44a86640f4a/
  inflating: chroma_db_tfm/chroma.sqlite3  
  inflating: chroma_db_tfm/71307125-5851-4ef3-957c-a44a86640f4a/data_level0.bin  
  inflating: chroma_db_tfm/71307125-5851-4ef3-957c-a44a86640f4a/index_metadata.pickle  
  inflating: chroma_db_tfm/71307125-5851-4ef3-957c-a44a86640f4a/header.bin  
  inflating: chroma_db_tfm/71307125-5851-4ef3-957c-a44a86640f4a/link_lists.bin  
  inflating: chroma_db_tfm/71307125-5851-4ef3-957c-a44a86640f4a/length.bin  
Cargando motor de búsqueda ligero...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  # Means the user did not define a `HF_TOKEN` secret => warn


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/tmp/ipykernel_33739/887705918.py:19: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


Sistema cargado con éxito usando FastEmbed.


In [28]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="llama3")

def preguntar_al_sistema(pregunta):
    docs = vector_db.similarity_search(pregunta, k=3)
    # Filtro para evitar el ValidationError de Pydantic que nos daba antes
    contextos = [d.page_content for d in docs if d and getattr(d, 'page_content', None)]
    contexto_final = "\n\n".join(contextos)

    prompt = f"Contexto: {contexto_final}\n\nPregunta: {pregunta}\n\nRespuesta:"
    return llm.invoke(prompt)

print("Sistema operativo.")

Sistema operativo.


In [29]:
import shutil
from google.colab import files

archivo_zip = "rag_tfm_final.zip"
shutil.make_archive("rag_tfm_final", 'zip', "./chroma_db_tfm")

files.download(f"{archivo_zip}")

print("Descarga iniciada.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada.
